# ZhiXia Comprehensive Testing Notebook

This notebook provides independent testing for each module:

- **Part 1: ASR** - Speech Recognition
- **Part 2: LLM** - LLM Inference  
- **Part 3: TTS** - Text-to-Speech
- **Part 4: Full Pipeline** (Optional)

## Usage
1. Run Cell 0: Environment Setup first
2. Then run any of the 4 parts independently


## Cell 0: Environment Configuration

In [ ]:
import os
import sys
import gc
import time
from pathlib import Path
from IPython.display import Audio, display
import ipywidgets as widgets
from ipywidgets import VBox, HBox, Output
try:
    import pandas as pd
except ImportError:
    pd = None

# Setup paths
NOTEBOOK_DIR = Path(".").absolute()
PROJECT_ROOT = NOTEBOOK_DIR.parent
sys.path.insert(0, str(PROJECT_ROOT))

# Setup environment variables
os.environ["MODELSCOPE_CACHE"] = str(PROJECT_ROOT / ".cache" / "modelscope")
os.environ["LD_LIBRARY_PATH"] = str(PROJECT_ROOT / "rknn_libs") + ":" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["HOME"] = str(PROJECT_ROOT)

# Create directories
for dir_name in ["models", "output", ".cache/modelscope"]:
    (PROJECT_ROOT / dir_name).mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"MODELSCOPE_CACHE: {os.environ["MODELSCOPE_CACHE"]}")

# Check key files
critical_files = {
    "rkllm_inference.py": PROJECT_ROOT / "rkllm_inference.py",
    "asr_llm_tts_npu_fast.py": PROJECT_ROOT / "asr_llm_tts_npu_fast.py",
    "asr_llm_tts_npu_only.py": PROJECT_ROOT / "asr_llm_tts_npu_only.py",
    "notebook_helpers.py": NOTEBOOK_DIR / "notebook_helpers.py",
}
for name, path in critical_files.items():
    print(f"  {"OK" if path.exists() else "MISSING"}: {name}")

# Import helpers
from notebook_helpers import (
    create_asr_ui, create_llm_ui, create_llm_inference_ui,
    create_tts_synthesis_ui, create_tts_comparison_ui, create_pipeline_ui
)
from asr_llm_tts_npu_fast import tts_synthesis_fast
from asr_llm_tts_npu_only import tts_synthesis as tts_synthesis_offline

# Global model variables
asr_model = None
llm_model = None

print("Environment setup complete.")

---
# Part 1: ASR Speech Recognition

Using FunASR Paraformer INT8 for Chinese speech recognition

In [ ]:
from funasr import AutoModel

asr_model = AutoModel(
    model="iic/speech_paraformer_asr_nat-zh-cn-16k-common-vocab8358-tensorflow1",
    vad_model=None,
    punc_model=None,
    disable_update=True,
    hub="ms",
    device="cpu",
)
print("ASR model loaded.")

In [ ]:
create_asr_ui(PROJECT_ROOT, asr_model)

---
# Part 2: LLM NPU Inference

Using RKLLM for Qwen model inference with NPU acceleration

In [ ]:
create_llm_ui(PROJECT_ROOT, llm_model)

In [ ]:
create_llm_inference_ui(llm_model)

---
# Part 3: TTS Text-to-Speech

Supporting both fast (MeloTTS) and offline (PaddleSpeech) versions

In [ ]:
create_tts_synthesis_ui(PROJECT_ROOT, tts_synthesis_fast, tts_synthesis_offline)

In [ ]:
create_tts_comparison_ui(PROJECT_ROOT, tts_synthesis_fast, tts_synthesis_offline)

In [ ]:
create_pipeline_ui(PROJECT_ROOT, asr_model, llm_model, tts_synthesis_fast, tts_synthesis_offline)